# TP Blanc — Détection de Maladie Rénale Chronique

**Durée :** 2h30  
**Documents :** Cours autorisé, Internet autorisé  
**Rendu :** Ce notebook complété (cellules de code + réponses en markdown)

---

## Contexte

La **maladie rénale chronique** (CKD — *Chronic Kidney Disease*) est une dégradation progressive et irréversible des reins. Elle touche environ 10% de la population mondiale et est souvent silencieuse jusqu'aux stades avancés.

Ce dataset a été collecté dans un hôpital de Tamil Nadu, Inde, sur environ 2 mois. Il contient des analyses biologiques et des antécédents médicaux de **400 patients**.

**Votre mission :** construire et évaluer un modèle de détection automatique de CKD à partir de données cliniques.

---

## Dictionnaire des Variables

| Abréviation | Signification | Unité / Valeurs |
|---|---|---|
| `age` | Âge | années |
| `bp` | Pression artérielle (*Blood Pressure*) | mm/Hg |
| `sg` | Gravité spécifique urine (*Specific Gravity*) | 1.005, 1.010, 1.015, 1.020, 1.025 |
| `al` | Albumine urinaire | 0–5 |
| `su` | Sucre urinaire (*Sugar*) | 0–5 |
| `rbc` | Globules rouges urine (*Red Blood Cells*) | normal / abnormal |
| `pc` | Cellules de pus (*Pus Cell*) | normal / abnormal |
| `pcc` | Amas de cellules de pus (*Pus Cell Clumps*) | present / notpresent |
| `ba` | Bactéries | present / notpresent |
| `bgr` | Glycémie aléatoire (*Blood Glucose Random*) | mgs/dl |
| `bu` | Urée sanguine (*Blood Urea*) | mgs/dl |
| `sc` | Créatinine sérique (*Serum Creatinine*) | mgs/dl |
| `sod` | Sodium | mEq/L |
| `pot` | Potassium | mEq/L |
| `hemo` | Hémoglobine | g/dL |
| `pcv` | Volume globulaire (*Packed Cell Volume*) | % |
| `wc` | Leucocytes (*White Blood Cell count*) | cellules/µL |
| `rc` | Érythrocytes (*Red Blood Cell count*) | millions/µL |
| `htn` | Hypertension | yes / no |
| `dm` | Diabète (*Diabetes Mellitus*) | yes / no |
| `cad` | Maladie coronarienne (*Coronary Artery Disease*) | yes / no |
| `appet` | Appétit | good / poor |
| `pe` | Œdème des chevilles (*Pedal Edema*) | yes / no |
| `ane` | Anémie | yes / no |
| `class` | **Cible** | ckd / notckd |

---

## ⚠️ Note sur les données

Ce dataset a été collecté dans un contexte hospitalier réel et contient des **problèmes de qualité importants**. Le bloc de chargement ci-dessous vous est fourni car certains problèmes sont spécifiques au format du fichier source (caractères `\t`, colonnes numériques stockées comme texte).

**Au-delà du preprocessing fourni, vous devrez gérer vous-mêmes les valeurs manquantes et l'encodage.**

In [ ]:
# !pip install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)
# Ajoutez vos imports selon les modèles choisis

np.random.seed(42)

---
## Chargement et Preprocessing de Base
*(Bloc fourni — lisez-le attentivement avant de continuer)*

In [ ]:
# --- CHARGEMENT ---
ckd = fetch_ucirepo(id=336)
df = pd.concat([ckd.data.features, ckd.data.targets], axis=1)

# --- NETTOYAGE DU FORMAT SOURCE ---
# Ce dataset contient des artefacts du format ARFF d'origine :
#   - valeurs manquantes encodées '?' ou '\t?'
#   - espaces et tabulations parasites dans les valeurs texte
#   - colonnes pcv / wc / rc stockées comme object malgré être numériques

# 1. Normaliser les chaînes : supprimer espaces et tabulations
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

# 2. Remplacer toutes les formes de valeurs manquantes par NaN
df.replace(['?', '\t?', 'nan', 'None', ''], np.nan, inplace=True)

# 3. Convertir les colonnes numériques mal typées
for col in ['pcv', 'wc', 'rc']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Cible en binaire
df['class'] = df['class'].map({'ckd': 1, 'notckd': 0})

print(f"Dataset chargé : {df.shape[0]} patients, {df.shape[1]} colonnes")
print(f"Types de colonnes : {df.dtypes.value_counts().to_dict()}")
df.head()

---
## Partie 1 — Exploration des Données  *(~30 min)*

**1.1** Affichez les statistiques descriptives du dataset. Combien y a-t-il de valeurs manquantes par colonne ? Identifiez les 3 colonnes les plus problématiques.

**1.2** Analysez la variable cible `class`. Ce problème est-il équilibré ? Quelle conséquence cela a-t-il sur le choix des métriques ?

**1.3** Visualisez la distribution de **3 features numériques** selon la classe (CKD vs non-CKD). Quelles features semblent les plus discriminantes visuellement ?

**1.4** Calculez et affichez la corrélation entre les features numériques et la cible. Commentez.

In [ ]:
# Partie 1

*Vos observations ici (double-cliquez pour éditer)*

---
## Partie 2 — Prétraitement Complet  *(~40 min)*

**2.1** Gérez les valeurs manquantes. Justifiez votre stratégie pour les colonnes numériques ET catégorielles (les deux n'ont pas forcément la même réponse).

**2.2** Encodez les variables catégorielles en valeurs numériques.

**2.3** Séparez le dataset en train/test. Justifiez vos choix (taille, stratification, seed). En vous souvenant des erreurs vues en TP7, y a-t-il un risque particulier ici ?

**2.4** Appliquez une normalisation. Pourquoi est-il important de faire cela **après** le split ?

In [ ]:
# Partie 2

*Vos justifications ici*

---
## Partie 3 — Modélisation  *(~45 min)*

**3.1** Entraînez **au moins deux modèles** de types différents (ex : modèle linéaire + modèle ensembliste). Vous êtes libres du choix des hyperparamètres.

**3.2** Choisissez vos métriques d'évaluation et justifiez ce choix dans le contexte médical de ce problème. L'accuracy suffit-elle ?

**3.3** Comparez vos modèles de manière rigoureuse (tableau récapitulatif + visualisations).

**3.4** Tracez les courbes ROC de vos deux modèles sur le même graphique. Interprétez.

In [ ]:
# Partie 3

*Vos justifications et interprétations ici*

---
## Partie 4 — Analyse et Interprétation  *(~35 min)*

**4.1** Identifiez les 5 features les plus importantes selon votre meilleur modèle. Ces résultats sont-ils cohérents avec ce que vous avez observé en Partie 1 ?

**4.2** Analysez les **erreurs** de votre modèle : combien de Faux Positifs ? de Faux Négatifs ? Dans ce contexte clinique (dépistage de maladie rénale), lequel est le plus dangereux ? Comment modifier votre modèle pour le minimiser ?

**4.3** Appliquez la correction identifiée en 4.2. Comparez les métriques avant/après.

**4.4** Rédigez une **conclusion de 5 à 8 phrases** qui répond aux questions suivantes :
- Votre modèle est-il prêt pour un déploiement hospitalier réel ?
- Quelles sont les 2 principales limites de cette étude ?
- Quelle prochaine étape recommanderiez-vous ?

In [ ]:
# Partie 4

*Votre conclusion ici*

---
## Bonus *(pour aller plus loin)*

**B.1** Essayez une imputation plus sophistiquée que la médiane/mode (ex : `KNNImputer` de sklearn). Est-ce que cela améliore les performances ?

**B.2** Effectuez une optimisation d'hyperparamètres (`GridSearchCV` ou `RandomizedSearchCV`) sur votre meilleur modèle.

**B.3** La colonne `dm` (diabète) contient des incohérences subtiles dans ses valeurs. Identifiez-les et corrigez-les.

In [ ]:
# Bonus